In [2]:
import pandas as pd
import numpy as np

dataset = pd.DataFrame()

In [ ]:
import urllib.request
import json
# TODO: Replace with your actual roll number
ROLL_NUMBER = 2025701040
INDEX = 100
url = f"http://preon.iiit.ac.in:8026/api/data?roll={ROLL_NUMBER}&index={INDEX}"
try:
    with urllib.request.urlopen(url) as response:
        if response.status == 200:
            data = json.loads(response.read().decode())
            print(f"Features: {data['features']}")
            print(f"Label: {data['label']}")
except urllib.error.HTTPError as e:
    if e.code == 404:
        print("End of dataset reached. Stop collecting.")
    else:
        print(f"An HTTP error occurred: {e.code}")

Features: [2.438138979641836, 0.9860516005825798, 1.0885612867536447, 1.1105189348414264, 10.882418234009728]
Label: 0


In [15]:
type(data['features'])

list

In [16]:
da = np.array(data['features'])

In [27]:
dataset = pd.concat([dataset, pd.DataFrame(da.reshape(1, -1), columns=[f'feature_{i}' for i in range(5)])], ignore_index=True)

In [ ]:
import urllib.request
import json
ROLL_NUMBER = 2025701040

for i in range(10000):  
    INDEX = i
    url = f"http://preon.iiit.ac.in:8026/api/data?roll={ROLL_NUMBER}&index={INDEX}"
    try:
        with urllib.request.urlopen(url) as response:
            if response.status == 200:
                data = json.loads(response.read().decode())

                print(f"Features: {data['features']}")
                print(f"Label: {data['label']}")
                with open('dataset.json', 'a') as f:
                    f.write(json.dumps(data) + '\n')
    except urllib.error.HTTPError as e:
        if e.code == 404:
            print("End of dataset reached. Stop collecting.")
            break
        else:
            print(f"An HTTP error occurred: {e.code}")

In [3]:
dataset = pd.read_json('dataset.json', lines=True)
dataset

,roll,index,features,label
0,2025701040,0,"[1.442314693770793, 7.144374835963619, 3.63936...",0
1,2025701040,1,"[22.71704929617661, 22.83002795011122, 8.35492...",1
2,2025701040,2,"[0.699387023845017, 3.950427855318255, 14.8869...",0
3,2025701040,3,"[60.272683831761825, 159.84915720099562, 8.070...",1
4,2025701040,4,"[10.981090631672586, 15.456586843717423, 111.0...",1
...,...,...,...,...
1996,2025701040,1996,"[2.3270333257435523, 0.6476945990681541, 1.710...",0
1997,2025701040,1997,"[9.907356423988084, 3.228012276338029, 1.54322...",0
1998,2025701040,1998,"[2.3385273607073582, 3.402183835995687, 1.5875...",0
1999,2025701040,1999,"[2.359380743381411, 3.035945006792189, 11.8843...",0


In [4]:
dataset['features'] = dataset['features'].apply(lambda x: np.array(x))
dataset

,roll,index,features,label
0,2025701040,0,"[1.442314693770793, 7.144374835963619, 3.63936...",0
1,2025701040,1,"[22.71704929617661, 22.83002795011122, 8.35492...",1
2,2025701040,2,"[0.699387023845017, 3.950427855318255, 14.8869...",0
3,2025701040,3,"[60.272683831761825, 159.84915720099562, 8.070...",1
4,2025701040,4,"[10.981090631672586, 15.456586843717423, 111.0...",1
...,...,...,...,...
1996,2025701040,1996,"[2.3270333257435523, 0.6476945990681541, 1.710...",0
1997,2025701040,1997,"[9.907356423988084, 3.228012276338029, 1.54322...",0
1998,2025701040,1998,"[2.3385273607073582, 3.402183835995687, 1.5875...",0
1999,2025701040,1999,"[2.359380743381411, 3.035945006792189, 11.8843...",0


In [18]:
SCENARIO_ID = ROLL_NUMBER% 8 
SCENARIO_ID

0

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [11]:
x = dataset['features'].array

In [7]:
seed  = 2025701040

In [8]:
np.random.seed(seed)

indices = np.random.permutation(len(dataset))
train_size = int(0.8 * len(dataset))
train_indices = indices[:train_size]
test_indices = indices[train_size:]
x_train = np.stack(dataset['features'].iloc[train_indices])
y_train = dataset['label'].iloc[train_indices].values
x_test = np.stack(dataset['features'].iloc[test_indices])
y_test = dataset['label'].iloc[test_indices].values

In [9]:
indices

array([1020, 1531, 1367, ...,  480, 1672, 1544], shape=(2001,))

In [10]:
x_train

array([[12.6814112 , 85.93145856, 32.42013102, 13.86668155, 15.63207965],
       [ 2.52616959,  9.35696566,  4.73079408,  4.66499081,  1.48735101],
       [ 7.31956579, 44.27019732, 14.08157688,  6.6059425 , 20.38520051],
       ...,
       [ 1.65325626,  7.34701643,  1.96023807,  2.49321348,  1.83692913],
       [ 6.60800044,  1.72149362,  5.32337591,  7.17831316,  2.04272116],
       [ 1.12537368,  1.73772623,  1.62328979,  1.93327155,  1.25834135]],
      shape=(1600, 5))

In [11]:
y_train

array([1, 0, 1, ..., 0, 0, 0], shape=(1600,))

In [12]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(x_train, y_train)
y_pred = knn.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9825


In [13]:

mean = np.mean(x_train, axis=0)
std = np.std(x_train, axis=0)
min = np.min(x_train, axis=0)
max = np.max(x_train, axis=0)
quantiles = np.percentile(x_train, [25, 50, 75], axis=0)
print("Mean:", mean)
print("Standard Deviation:", std)
print("Min:", min)
print("Max:", max)
print("Quantiles (25th, 50th, 75th):", quantiles)

Mean: [19.24562545 17.54991257 19.51936908 17.49126467 18.4835133 ]
Standard Deviation: [41.38497496 31.60590806 34.1740437  30.7708167  31.67066877]
Min: [0.14286944 0.08730689 0.03619444 0.12709247 0.05635146]
Max: [1012.92609396  590.36883898  414.56551687  446.35463834  418.14829238]
Quantiles (25th, 50th, 75th): [[ 2.59098114  2.57058776  2.62997424  2.50091387  2.49166946]
 [ 7.26707811  6.88989417  6.98859795  7.15162667  6.98493632]
 [20.50023169 19.10868698 21.67237926 19.87242914 20.56808485]]
